In [1]:
using Clapeyron, Metaheuristics, Printf

In [2]:
#Hfus dan Tm masukin ke csv yh jgn lupa

like_parameter = """Clapeyron Database File
PCSAFT Like Parameters [csvtype = like,grouptype = PCSAFT]
species,Mw,segment,sigma,epsilon,n_H,n_e
serine,105.09,7.0236,2.2840,236.9200,2,2
water,18.02,1.2047,2.801457,353.94,1,1
"""

unlike_parameter = """Clapeyron Database File
PCSAFT Unlike Parameters [csvtype = unlike,grouptype = PCSAFT]
species1,species2,k
water,serine,-0.0257
"""

#jgn lupa combing rules yh
assoc_parameter = """Clapeyron Database File
PCSAFT Assoc Parameters [csvtype = assoc,grouptype = PCSAFT]
species1,site1,species2,site2,epsilon_assoc,bondvol
water,H,water,e,2425.67,0.045
serine,H,serine,e,2671.93,0.0385
water,H,serine,e,2548.8,0.041407276
water,e,serine,H,2548.8,0.041407276
"""
components = ["water", "serine"]
model = CompositeModel(components;
                       fluid = PCSAFT,
                       solid = SolidHfus,
                       fluid_userlocations = [like_parameter, unlike_parameter, assoc_parameter])

println(model.fluid.params.epsilon.values)
println(model.fluid.params.sigma.values)
println("======================")
println(model.fluid.params.epsilon_assoc.values)
println(model.fluid.params.bondvol.values)
println("kij = ", (1  - ((model.fluid.params.epsilon.values[2])/(sqrt(model.fluid.params.epsilon.values[1] * model.fluid.params.epsilon.values[4])))))

[353.94 297.0202373352122; 297.0202373352122 236.92]
[2.8014570000000003e-10 2.5427285e-10; 2.5427285e-10 2.2839999999999998e-10]
Clapeyron.Compressed4DMatrix{Float64, Vector{Float64}}[2425.67, 2548.8, 2548.8, 2671.93]
Clapeyron.Compressed4DMatrix{Float64, Vector{Float64}}[0.045, 0.041407276, 0.041407276, 0.0385]
kij = -0.025700000000000056


In [3]:
# Run this ONCE to fix your CSV files
function fix_line_endings(filename)
    content = read(filename, String)
    fixed = replace(content, "\r\n" => "\n")
    write(filename, fixed)
    println("Fixed: $filename")
end

fix_line_endings("gamma_serine.csv")
fix_line_endings("rhol_serine.csv")

Fixed: gamma_serine.csv
Fixed: rhol_serine.csv


In [4]:
#SOLUTION DENSITY

Mw_water = model.fluid.params.Mw[1] / 1000
Mw_aa = model.fluid.params.Mw[2] / 1000

println("MW Solute  : ", Mw_aa, " kg/m3")
println("MW solvent : ", Mw_water, " kg/m3")

function molality_to_z(m::Float64, Mw_solute::Float64)
    # m mol solute per kg water
    n_solute = m
    n_water  = 1.0 / Mw_water   # mol water per kg water = 55.51
    x_water  = n_water / (n_water + n_solute)
    x_solute = n_solute / (n_water + n_solute)
    return [x_water, x_solute]   # [water, amino acid]
end

function solution_density(model::EoSModel, m::Float64)
    z   = molality_to_z(m, Mw_aa)
    T   = 298.15
    p   = 101325.0
    V   = volume(model, p, T, z; phase=:liquid)        # m³/mol mixture
    Mw_mix = z[1]*Mw_water + z[2]*Mw_aa               # kg/mol
    return Mw_mix / V                                   # kg/m³
end

MW Solute  : 0.10509 kg/m3
MW solvent : 0.01802 kg/m3


solution_density (generic function with 1 method)

In [5]:
#ACTIVITY COEFFICIENT

function chem_activity_coeff(model::EoSModel, m::Float64)
    z     = molality_to_z(m, model.fluid.params.Mw[2])
    z_inf = molality_to_z(1e-10, model.fluid.params.Mw[2])   # very dilute reference
    p     = 101325.0
    T     = 298.15
    RT    = Rgas(model) * T

    # get chemical potentials directly
    chem_pot     = chemical_potential(model.fluid, p, T, z;     phase=:liquid)
    chem_pot_inf = chemical_potential(model.fluid, p, T, z_inf; phase=:liquid)

    # γ dari persamaan chemical potential
    gamma_star_x = exp((chem_pot[2] - chem_pot_inf[2]) / RT) * (z_inf[2] / z[2])

    # convert to molality basis (Kuramochi eq 5)
    #println("exp((μ[2] - μ_inf[2]) / RT) :", exp((μ[2] - μ_inf[2]) / RT))
    #println("m :",m)
    #println("z_inf :",z_inf)
    #println("z[2]: ",z[2])
    #println("z_inf[2] / z[2] :",z_inf[2] / z[2])
    return gamma_star_x / (1.0 + Mw_water * m)
end

chem_activity_coeff (generic function with 1 method)

In [6]:
toestimate = [
    Dict(
        :param => :epsilon,
        :indices => (2,2),
        :lower => 200.,
        :upper => 400.,
        :guess => 350.
    ),
    Dict(
        :param => :sigma,
        :indices => (2,2),
        :factor => 1e-10,
        :lower => 2.0,
        :upper => 3.0,
        :guess => 2.3
    ),
    Dict(
        :param   => :epsilon_assoc,
        :indices => 4,
        #:cross_assoc => true,
        :lower   => 2000.0,
        :upper   => 3000.0,
        :guess   => 2400.0
    ),
    Dict(
        :param   => :bondvol,
        :indices => 4,
        #:cross_assoc => true,
        :lower   => 0.03,
        :upper   => 0.04,
        :guess   => 0.038
    ),
]

4-element Vector{Dict{Symbol, Any}}:
 Dict(:upper => 400.0, :param => :epsilon, :indices => (2, 2), :guess => 350.0, :lower => 200.0)
 Dict(:factor => 1.0e-10, :param => :sigma, :indices => (2, 2), :upper => 3.0, :lower => 2.0, :guess => 2.3)
 Dict(:upper => 3000.0, :param => :epsilon_assoc, :indices => 4, :guess => 2400.0, :lower => 2000.0)
 Dict(:upper => 0.04, :param => :bondvol, :indices => 4, :guess => 0.038, :lower => 0.03)

In [7]:
estimator, objective, x0, upper, lower = Estimation(
    model,
    toestimate,
    [
        "rhol_serine.csv",
        "gamma_serine.csv",
    ]
)
 
println("Initial objective value: ", objective(x0))
println(x0)

Initial objective value: 0.270378829526316
[350.0, 2.3, 2400.0, 0.038]


In [8]:
method = ECA(; options = Options(iterations = 10000, seed = 42))
 
params_opt, model_opt = optimize(objective, estimator, method)

([219.8873612048151, 2.000000000000019, 2000.000000000436, 0.030000000000173208], CompositeModel{PCSAFT{BasicIdeal, Float64}, SolidHfus}("water", "serine"))

In [9]:
println("\n=== Optimized PC-SAFT parameters for glycine ===")
println("  segment (m)  : ", model_opt.fluid.params.segment[2])
println("  sigma   (m)  : ", model_opt.fluid.params.sigma[2,2])
println("  epsilon (K1)  : ", model_opt.fluid.params.epsilon.values)
println("  epsilon_assoc  : ", model_opt.fluid.params.epsilon_assoc)
println("  bondvol  : ", model_opt.fluid.params.bondvol)
println("==================")
println(model_opt.fluid.params.epsilon.values)
println(model_opt.fluid.params.sigma.values)
println("kij = ", (1  - ((model_opt.fluid.params.epsilon.values[2])/(sqrt(model_opt.fluid.params.epsilon.values[1] * model_opt.fluid.params.epsilon.values[4])))))
println("======================")
println(model_opt.fluid.params.epsilon_assoc.values)
println(model_opt.fluid.params.bondvol.values)



=== Optimized PC-SAFT parameters for glycine ===
  segment (m)  : 7.0236
  sigma   (m)  : 2.0000000000000192e-10
  epsilon (K1)  : [353.94 297.0202373352122; 297.0202373352122 219.8873612048151]
  epsilon_assoc  : AssocParam{Float64}("epsilon_assoc")[2425.67, 2548.8, 2548.8, 2000.000000000436]
  bondvol  : AssocParam{Float64}("bondvol")[0.045, 0.041407276, 0.041407276, 0.030000000000173208]
[353.94 297.0202373352122; 297.0202373352122 219.8873612048151]
[2.8014570000000003e-10 2.40072850000001e-10; 2.40072850000001e-10 2.0000000000000192e-10]
kij = -0.06468487321160699
Clapeyron.Compressed4DMatrix{Float64, Vector{Float64}}[2425.67, 2548.8, 2548.8, 2000.000000000436]
Clapeyron.Compressed4DMatrix{Float64, Vector{Float64}}[0.045, 0.041407276, 0.041407276, 0.030000000000173208]


In [10]:
using CSV, DataFrames, Printf

function calculate_AAD(model, csv_file, property_func)
    df = CSV.read(csv_file, DataFrame, comment="#", skipto=4)
    
    input_col  = names(df)[1]          # first column = input (T)
    output_col = names(df)[2]          # second column = out_xxx (experimental)
    
    inputs   = df[!, input_col]
    exp_vals = df[!, output_col]
    
    println("\n=== AAD: $csv_file ===")
    @printf("%-10s  %-12s  %-12s  %-8s\n", input_col, "exp", "calc", "ARD%")
    
    errors = Float64[]
    for (i, x) in enumerate(inputs)
        calc = property_func(model, x)
        err  = abs(calc - exp_vals[i]) / abs(exp_vals[i]) * 100
        push!(errors, err)
        @printf("%-10.4f  %-12.6f  %-12.6f  %-8.4f\n", x, exp_vals[i], calc, err)
    end
    
    aard = sum(errors) / length(errors)
    @printf("AARD = %.4f%%\n", aard)
    return aard
end

calculate_AAD (generic function with 1 method)

In [11]:
aard_p   = calculate_AAD(model_opt, "rhol_serine.csv", solution_density)

┌ Warning: thread = 1 warning: parsed expected 1 columns, but didn't reach end of line around data row: 1. Parsing extra columns and widening final columnset
└ @ CSV C:\Users\sutha\.julia\packages\CSV\XLcqT\src\file.jl:593



=== AAD: rhol_serine.csv ===
Clapeyron Estimator  exp           calc          ARD%    
0.0297      1001.264000   994.456555    0.6799  
0.0652      1002.818000   996.751895    0.6049  
0.1007      1004.520000   999.040911    0.5454  
0.1519      1006.826000   1002.332480   0.4463  
0.2039      1009.142000   1005.664805   0.3446  
0.2524      1011.312000   1008.763157   0.2520  
0.3016      1013.500000   1011.896696   0.1582  
0.3506      1015.667000   1015.007969   0.0649  
AARD = 0.3870%


0.38702799206647653

In [12]:
aard_p   = calculate_AAD(model_opt, "gamma_serine.csv", chem_activity_coeff)


=== AAD: gamma_serine.csv ===
Clapeyron Estimator  exp           calc          ARD%    
0.0543      0.967500      0.967917      0.0431  
0.1515      0.914300      0.915698      0.1529  
0.2504      0.867200      0.868633      0.1653  
0.3657      0.819400      0.820416      0.1240  
0.4015      0.805900      0.806739      0.1042  
0.4665      0.782800      0.783329      0.0676  
0.5011      0.771300      0.771571      0.0351  
0.9047      0.665300      0.663746      0.2336  
AARD = 0.1157%


┌ Warning: thread = 1 warning: parsed expected 1 columns, but didn't reach end of line around data row: 1. Parsing extra columns and widening final columnset
└ @ CSV C:\Users\sutha\.julia\packages\CSV\XLcqT\src\file.jl:593


0.11571018090875992